# PS 6 — Online Retail (UCI)

**DWM practical | Run cells top-to-bottom.**

All datasets are read from `../data/` and results saved in `../outputs/`. Run `python download_datasets.py` in the repo root beforehand.

**Goal:** load UCI Excel data, parse invoice dates, handle missing/inconsistent/duplicate records, encode a few categorical variables and normalize quantity and unit price. Cancelled invoices (prefix `C`) and negative/zero quantity or unit price are excluded from the clean sales subset.**

The workbook is large; `download_datasets.py --retail` downloads it once.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
DATA = Path('../data')
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)

import warnings
file = DATA/'Online Retail.xlsx'
if not file.exists(): raise FileNotFoundError('Run: python download_datasets.py --retail')
df = pd.read_excel(file,engine='openpyxl')
before = df.copy()

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:'); print(df.dtypes)
print('Missing values:'); print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())
display(df.head())
display(df.describe(include='all').T)

In [ ]:
print('Data types BEFORE:',df.dtypes)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'],errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'],errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'],errors='coerce')
for c in ['InvoiceNo','StockCode','Description','Country']:
    df[c] = df[c].astype('string').str.strip()
print('Duplicate rows BEFORE:',df.duplicated().sum())
print('Missing BEFORE:',df.isna().sum())

**Quality rules:** remove exact duplicates, invalid dates, missing transaction identifiers/descriptions/country, cancellations, nonpositive quantity and nonpositive price. Missing CustomerID may represent guest purchases, so keep it as nullable `Int64` rather than discarding otherwise valid transactions.

In [ ]:
df = df.drop_duplicates().copy()
df = df.dropna(subset=['InvoiceNo','StockCode','Description','Country','InvoiceDate','Quantity','UnitPrice'])
df = df[(df['InvoiceNo']!='') & (df['Description']!='') & (df['Country']!='')]
df = df[~df['InvoiceNo'].str.upper().str.startswith('C',na=False)]
df = df[(df['Quantity']>0) & (df['UnitPrice']>0)].copy()
df['CustomerID'] = pd.to_numeric(df['CustomerID'],errors='coerce').astype('Int64')
df['Country'] = df['Country'].str.strip().str.title().astype('category')
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
# Encode a selected small nominal category rather than all thousands of product identifiers.
df['IsUnitedKingdom'] = (df['Country'].astype('string')=='United Kingdom').astype(int)
df['Revenue'] = df['Quantity']*df['UnitPrice']
print('Shape after cleaning:',df.shape)
print('Nulls after cleaning:',df.isna().sum())
display(df.head())

**Encoding:** UK indicator is 1 for UK, 0 elsewhere; month is derived from datetime. In a real model, encode other countries as needed. Scale `Quantity` and `UnitPrice` into two new columns, preserving original amounts so revenue remains meaningful.

In [ ]:
scale_cols=['Quantity','UnitPrice']
print('Before scaling:');display(df[scale_cols].head())
scaled=MinMaxScaler().fit_transform(df[scale_cols])
df[['Quantity_scaled','UnitPrice_scaled']]=scaled
print('After scaling:');display(df[['Quantity_scaled','UnitPrice_scaled']].head())
print('Date type:',df['InvoiceDate'].dtype)
print('Duplicate rows after:',df.duplicated().sum())
print('Before sample:');display(before.head())
print('After sample:');display(df.head())

In [ ]:
df.to_csv(OUT/'PS06_online_retail_processed.csv',index=False)
print('Saved cleaned retail records:',len(df))